In [1]:
import json, glob, re, pycm, pandas as pd, numpy as np, scipy.stats as stats

In [2]:
RUN_VERSION = "v29"

In [3]:
data = []
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    df = pd.DataFrame(json.load(open(file, 'r')))
    keys = {
        "Judge Model": model,
        "Prompt": prompt,
        "Dataset": dataset
    }
    n = len(df)
    wk_v_counts = df["wk_v"].value_counts()
    wk_v_freq = (wk_v_counts / n).to_dict()
    I_freq = {
        '<t,t>': 0.0, 
        '<t,e>': 0.0, 
        '<t,f>': 0.0, 
        '<e,t>': 0.0, 
        '<e,e>': 0.0, 
        '<e,f>': 0.0, 
        '<f,t>': 0.0, 
        '<f,e>': 0.0, 
        '<f,f>': 0.0
    }
    I_counts = df["I"].value_counts()
    I_freq_rec = (I_counts / n).to_dict()
    for tv in I_freq_rec:
        I_freq[tv] = I_freq_rec[tv]
    addl_stats = { 
        'Time (mean)': df['execution_time'].mean(),
        'Time (stdev)': df['execution_time'].std(),
        'Tokens (mean)': df['tokens_used'].mean(),
        'Tokens (stdev)': df['tokens_used'].std(),
        'Coverage': (n - wk_v_counts['e']) / n
    }
    data.append({ **keys, **wk_v_freq, **I_freq, **addl_stats })
df1 = pd.DataFrame.from_records(data)
df1 = df1.round(3)
df1 = df1.sort_values(["Judge Model", "Dataset", "Prompt"])
df1

,Judge Model,Prompt,Dataset,e,f,t,"<t,t>","<t,e>","<t,f>","<e,t>","<e,e>","<e,f>","<f,t>","<f,e>","<f,f>",Time (mean),Time (stdev),Tokens (mean),Tokens (stdev),Coverage
33,claude-3-5-haiku-20241022,baseline,gpqa,0.16,0.25,0.59,0.08,0.0,0.59,0.0,0.0,0.0,0.25,0.0,0.08,31.859,4.485,2642.59,648.881,0.84
14,claude-3-5-haiku-20241022,few,gpqa,0.53,0.19,0.28,0.48,0.0,0.28,0.0,0.0,0.0,0.19,0.0,0.05,46.959,4.305,7656.06,648.157,0.47
19,claude-3-5-haiku-20241022,zero,gpqa,0.58,0.19,0.23,0.50,0.0,0.23,0.0,0.0,0.0,0.19,0.0,0.08,43.194,3.771,4216.20,660.958,0.42
34,claude-3-5-haiku-20241022,baseline,simpleqa,0.52,0.18,0.30,0.01,0.0,0.30,0.0,0.0,0.0,0.18,0.0,0.51,15.979,3.291,1009.33,163.143,0.48
26,claude-3-5-haiku-20241022,few,simpleqa,0.61,0.21,0.18,0.14,0.0,0.18,0.0,0.0,0.0,0.21,0.0,0.47,41.053,4.229,6438.93,179.820,0.39
31,claude-3-5-haiku-20241022,zero,simpleqa,0.69,0.20,0.11,0.18,0.0,0.11,0.0,0.0,0.0,0.20,0.0,0.51,39.042,3.166,3107.98,143.984,0.31
24,claude-3-5-sonnet-20241022,baseline,gpqa,0.23,0.34,0.43,0.19,0.0,0.43,0.0,0.0,0.0,0.34,0.0,0.04,39.542,6.273,2965.82,765.463,0.77
4,claude-3-5-sonnet-20241022,few,gpqa,0.44,0.33,0.23,0.43,0.0,0.23,0.0,0.0,0.0,0.33,0.0,0.01,60.267,5.255,8043.21,702.081,0.56
17,claude-3-5-sonnet-20241022,zero,gpqa,0.49,0.30,0.21,0.45,0.0,0.21,0.0,0.0,0.0,0.30,0.0,0.04,61.849,5.505,4848.79,704.905,0.51
18,claude-3-5-sonnet-20241022,baseline,simpleqa,0.47,0.30,0.23,0.04,0.0,0.23,0.0,0.0,0.0,0.30,0.0,0.43,23.781,5.082,1197.56,187.527,0.53


In [4]:
cms = {}
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms:
        cms[model] = {}
    if prompt not in cms[model]:
        cms[model][prompt] = {}
    df = pd.DataFrame.from_records(json.load(open(file, 'r')))
    cms[model][prompt][dataset] = pycm.ConfusionMatrix(df["label"].tolist(), df["wk_v"].tolist(), digit=2, classes=[ 't', 'f' ])

data = [
    [ 
        model, 
        prompt, 
        dataset, 
        cms[model][prompt][dataset].F1_Macro, 
        cms[model][prompt][dataset].ACC_Macro, 
        cms[model][prompt][dataset].FPR['t'], 
        cms[model][prompt][dataset].FNR['t'], 
        cms[model][prompt][dataset].F1['t'], 
        cms[model][prompt][dataset].F1['f']
    ] 
    for model in cms 
    for prompt in cms[model] 
    for dataset in cms[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Macro-F1", "Acc.", "FPR", "FNR", "F1 (+)", "F1 (-)"]
df2 = pd.DataFrame(data, columns=column_names)
df2 = df2.round(3)
df2 = df2.sort_values(["Judge Model", "Dataset", "Prompt"])
df2

,Judge Model,Prompt,Dataset,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-)
34,claude-3-5-haiku-20241022,baseline,gpqa,0.562,0.571,0.617,0.189,0.625,0.500
30,claude-3-5-haiku-20241022,few,gpqa,0.638,0.638,0.464,0.211,0.638,0.638
32,claude-3-5-haiku-20241022,zero,gpqa,0.660,0.667,0.414,0.154,0.611,0.708
35,claude-3-5-haiku-20241022,baseline,simpleqa,0.666,0.667,0.464,0.150,0.680,0.652
31,claude-3-5-haiku-20241022,few,simpleqa,0.606,0.615,0.375,0.400,0.545,0.667
33,claude-3-5-haiku-20241022,zero,simpleqa,0.689,0.710,0.211,0.417,0.609,0.769
23,claude-3-5-sonnet-20241022,baseline,gpqa,0.701,0.701,0.366,0.222,0.709,0.693
18,claude-3-5-sonnet-20241022,few,gpqa,0.708,0.714,0.226,0.360,0.667,0.750
20,claude-3-5-sonnet-20241022,zero,gpqa,0.717,0.725,0.233,0.333,0.667,0.767
22,claude-3-5-sonnet-20241022,baseline,simpleqa,0.866,0.868,0.103,0.167,0.851,0.881


In [5]:
df = df2.merge(df1, on=['Judge Model', 'Prompt', 'Dataset'], how='inner')
df

,Judge Model,Prompt,Dataset,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-),e,...,"<e,e>","<e,f>","<f,t>","<f,e>","<f,f>",Time (mean),Time (stdev),Tokens (mean),Tokens (stdev),Coverage
0,claude-3-5-haiku-20241022,baseline,gpqa,0.562,0.571,0.617,0.189,0.625,0.500,0.16,...,0.0,0.0,0.25,0.0,0.08,31.859,4.485,2642.59,648.881,0.84
1,claude-3-5-haiku-20241022,few,gpqa,0.638,0.638,0.464,0.211,0.638,0.638,0.53,...,0.0,0.0,0.19,0.0,0.05,46.959,4.305,7656.06,648.157,0.47
2,claude-3-5-haiku-20241022,zero,gpqa,0.660,0.667,0.414,0.154,0.611,0.708,0.58,...,0.0,0.0,0.19,0.0,0.08,43.194,3.771,4216.20,660.958,0.42
3,claude-3-5-haiku-20241022,baseline,simpleqa,0.666,0.667,0.464,0.150,0.680,0.652,0.52,...,0.0,0.0,0.18,0.0,0.51,15.979,3.291,1009.33,163.143,0.48
4,claude-3-5-haiku-20241022,few,simpleqa,0.606,0.615,0.375,0.400,0.545,0.667,0.61,...,0.0,0.0,0.21,0.0,0.47,41.053,4.229,6438.93,179.820,0.39
5,claude-3-5-haiku-20241022,zero,simpleqa,0.689,0.710,0.211,0.417,0.609,0.769,0.69,...,0.0,0.0,0.20,0.0,0.51,39.042,3.166,3107.98,143.984,0.31
6,claude-3-5-sonnet-20241022,baseline,gpqa,0.701,0.701,0.366,0.222,0.709,0.693,0.23,...,0.0,0.0,0.34,0.0,0.04,39.542,6.273,2965.82,765.463,0.77
7,claude-3-5-sonnet-20241022,few,gpqa,0.708,0.714,0.226,0.360,0.667,0.750,0.44,...,0.0,0.0,0.33,0.0,0.01,60.267,5.255,8043.21,702.081,0.56
8,claude-3-5-sonnet-20241022,zero,gpqa,0.717,0.725,0.233,0.333,0.667,0.767,0.49,...,0.0,0.0,0.30,0.0,0.04,61.849,5.505,4848.79,704.905,0.51
9,claude-3-5-sonnet-20241022,baseline,simpleqa,0.866,0.868,0.103,0.167,0.851,0.881,0.47,...,0.0,0.0,0.30,0.0,0.43,23.781,5.082,1197.56,187.527,0.53


In [6]:
df_evaluation = df[["Dataset", "Judge Model", "Prompt", "Coverage", "Macro-F1", "Acc.", "FPR", "FNR", "F1 (+)", "F1 (-)"]].copy()
df_evaluation = df_evaluation.sort_values(["Dataset", "Judge Model", "Prompt"])
df_evaluation

,Dataset,Judge Model,Prompt,Coverage,Macro-F1,Acc.,FPR,FNR,F1 (+),F1 (-)
0,gpqa,claude-3-5-haiku-20241022,baseline,0.84,0.562,0.571,0.617,0.189,0.625,0.500
1,gpqa,claude-3-5-haiku-20241022,few,0.47,0.638,0.638,0.464,0.211,0.638,0.638
2,gpqa,claude-3-5-haiku-20241022,zero,0.42,0.660,0.667,0.414,0.154,0.611,0.708
6,gpqa,claude-3-5-sonnet-20241022,baseline,0.77,0.701,0.701,0.366,0.222,0.709,0.693
7,gpqa,claude-3-5-sonnet-20241022,few,0.56,0.708,0.714,0.226,0.360,0.667,0.750
8,gpqa,claude-3-5-sonnet-20241022,zero,0.51,0.717,0.725,0.233,0.333,0.667,0.767
12,gpqa,llama-4-maverick,baseline,0.85,0.786,0.788,0.255,0.147,0.763,0.809
13,gpqa,llama-4-maverick,few,0.83,0.771,0.771,0.262,0.195,0.776,0.765
14,gpqa,llama-4-maverick,zero,0.62,0.773,0.790,0.195,0.238,0.711,0.835
18,gpqa,llama-4-scout,baseline,0.72,0.763,0.764,0.250,0.219,0.746,0.779


In [7]:
df_evaluation.mean(numeric_only=True)

Coverage    0.581389
Macro-F1    0.647806
Acc.        0.677333
FPR         0.322056
FNR         0.347889
F1 (+)      0.613250
F1 (-)      0.682472
dtype: float64

In [8]:
df_evaluation[df_evaluation["Dataset"] == "gpqa"].mean(numeric_only=True)

Coverage    0.616111
Macro-F1    0.645333
Acc.        0.678944
FPR         0.280278
FNR         0.387333
F1 (+)      0.571389
F1 (-)      0.719278
dtype: float64

In [9]:
df_evaluation[df_evaluation["Dataset"] == "simpleqa"].mean(numeric_only=True)

Coverage    0.546667
Macro-F1    0.650278
Acc.        0.675722
FPR         0.363833
FNR         0.308444
F1 (+)      0.655111
F1 (-)      0.645667
dtype: float64

In [10]:
evaluation_table_latex = df_evaluation.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:0.3g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Evaluation of different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
    label="tab:expeval",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(evaluation_table_latex, file=open("paper/evaluation_table.tex", "w+"))


In [11]:
df_truth_value_distribution = df[["Dataset", "Judge Model", "Prompt", "Macro-F1", '<t,t>', '<t,f>', '<f,t>', '<f,f>']].copy()
df_truth_value_distribution = df_truth_value_distribution.sort_values(["Dataset", "Judge Model", "Prompt"])
df_truth_value_distribution

,Dataset,Judge Model,Prompt,Macro-F1,"<t,t>","<t,f>","<f,t>","<f,f>"
0,gpqa,claude-3-5-haiku-20241022,baseline,0.562,0.08,0.59,0.25,0.08
1,gpqa,claude-3-5-haiku-20241022,few,0.638,0.48,0.28,0.19,0.05
2,gpqa,claude-3-5-haiku-20241022,zero,0.660,0.50,0.23,0.19,0.08
6,gpqa,claude-3-5-sonnet-20241022,baseline,0.701,0.19,0.43,0.34,0.04
7,gpqa,claude-3-5-sonnet-20241022,few,0.708,0.43,0.23,0.33,0.01
8,gpqa,claude-3-5-sonnet-20241022,zero,0.717,0.45,0.21,0.30,0.04
12,gpqa,llama-4-maverick,baseline,0.786,0.05,0.42,0.43,0.10
13,gpqa,llama-4-maverick,few,0.771,0.13,0.44,0.39,0.04
14,gpqa,llama-4-maverick,zero,0.773,0.35,0.24,0.38,0.03
18,gpqa,llama-4-scout,baseline,0.763,0.07,0.35,0.37,0.21


In [12]:
df_truth_value_distribution.mean(numeric_only=True)

Macro-F1    0.647806
<t,t>       0.302222
<t,f>       0.284444
<f,t>       0.296944
<f,f>       0.116389
dtype: float64

In [13]:
df_truth_value_distribution.std(numeric_only=True)

Macro-F1    0.124073
<t,t>       0.194218
<t,f>       0.165313
<f,t>       0.115655
<f,f>       0.145409
dtype: float64

In [14]:
df_truth_value_distribution[df_truth_value_distribution["Dataset"] == "gpqa"].mean(numeric_only=True)

Macro-F1    0.645333
<t,t>       0.320556
<t,f>       0.271111
<f,t>       0.345000
<f,f>       0.063333
dtype: float64

In [15]:
df_truth_value_distribution[df_truth_value_distribution["Dataset"] == "simpleqa"].mean(numeric_only=True)

Macro-F1    0.650278
<t,t>       0.283889
<t,f>       0.297778
<f,t>       0.248889
<f,f>       0.169444
dtype: float64

In [16]:
truth_value_distribution_table_latex = df_truth_value_distribution.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:0.3g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Distribution of bilateral truth values different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
    label="tab:exptvdist",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(truth_value_distribution_table_latex, file=open("paper/truth_value_distribution_table.tex", "w+"))

In [17]:
df_cost = df[["Dataset", "Judge Model", "Prompt", 'Time (mean)', 'Time (stdev)', 'Tokens (mean)', 'Tokens (stdev)']].copy()
df_cost["Mean Time"] = df_cost["Time (mean)"].combine(df_cost["Time (stdev)"], lambda mean, sd: f'{mean:6.2f} ({sd:.2f})')
df_cost["Mean Tokens Used"] = df_cost["Tokens (mean)"].combine(df_cost["Tokens (stdev)"], lambda mean, sd: f'{mean:6.2f} ({sd:.2f})')
df_cost = df_cost[["Dataset", "Judge Model", "Prompt", 'Mean Time', 'Mean Tokens Used']]
df_cost = df_cost.sort_values(["Dataset", "Judge Model", "Prompt"])
df_cost

,Dataset,Judge Model,Prompt,Mean Time,Mean Tokens Used
0,gpqa,claude-3-5-haiku-20241022,baseline,31.86 (4.49),2642.59 (648.88)
1,gpqa,claude-3-5-haiku-20241022,few,46.96 (4.30),7656.06 (648.16)
2,gpqa,claude-3-5-haiku-20241022,zero,43.19 (3.77),4216.20 (660.96)
6,gpqa,claude-3-5-sonnet-20241022,baseline,39.54 (6.27),2965.82 (765.46)
7,gpqa,claude-3-5-sonnet-20241022,few,60.27 (5.25),8043.21 (702.08)
8,gpqa,claude-3-5-sonnet-20241022,zero,61.85 (5.50),4848.79 (704.90)
12,gpqa,llama-4-maverick,baseline,87.65 (96.52),6203.86 (1411.93)
13,gpqa,llama-4-maverick,few,75.69 (71.19),9906.28 (1393.11)
14,gpqa,llama-4-maverick,zero,77.48 (54.37),7425.84 (1237.05)
18,gpqa,llama-4-scout,baseline,64.16 (29.91),5425.48 (1792.92)


In [18]:
df_cost_numeric = df[["Dataset", "Judge Model", "Prompt", 'Time (mean)', 'Time (stdev)', 'Tokens (mean)', 'Tokens (stdev)']].copy()

In [19]:
df_cost_numeric.mean(numeric_only=True)

Time (mean)         40.041528
Time (stdev)        18.510750
Tokens (mean)     4928.956389
Tokens (stdev)     705.641306
dtype: float64

In [20]:
df_cost_numeric[df_cost_numeric["Judge Model"] == "nf-gpt-4o"].mean(numeric_only=True)

Time (mean)         18.685833
Time (stdev)         6.757500
Tokens (mean)     4354.358333
Tokens (stdev)     633.183500
dtype: float64

In [21]:
(300. * 40. * 3. * 3. * 3.) / (24. * 60. * 60.) # days to process

3.75

In [22]:
(300 * 4354 * 1 * 3 * 3) # total tokens

11755800

In [23]:
cost_table_latex = df_cost.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:6.6g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Execution time and tokens used by different judge models and evaluation prompts on three datasets (N=100).",        # Add a caption to your table
    label="tab:expcosts",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(cost_table_latex, file=open("paper/cost_table.tex", "w+"))

In [24]:
cms_wk = {}
cms_upper = {}
cms_lower = {}
choices = ["t", "f"]
for file in glob.glob(f"experiments/{RUN_VERSION}/*.json"):
    match = re.match(r'experiments/.+/(.+)-(baseline|zero|few)-(gpqa|simpleqa|mmlu-pro).json', file)
    model = match.group(1)
    prompt = match.group(2)
    dataset = match.group(3)
    if model not in cms_wk:
        cms_wk[model] = {}
    if prompt not in cms_wk[model]:
        cms_wk[model][prompt] = {}
    if model not in cms_upper:
        cms_upper[model] = {}
    if prompt not in cms_upper[model]:
        cms_upper[model][prompt] = {}
    if model not in cms_lower:
        cms_lower[model] = {}
    if prompt not in cms_lower[model]:
        cms_lower[model][prompt] = {}
    df = pd.DataFrame.from_records(json.load(open(file, 'r')))
    upper_conditions = [ (df["I_0"] == "t") & (df["I_1"] != "e"), (df["I_0"] == "f") & (df["I_1"] != "e") ]
    lower_conditions = [ (df["I_1"] == "f") & (df["I_0"] != "e"), (df["I_1"] == "t") & (df["I_0"] != "e") ]
    df["wk_v_upper"] = np.select(upper_conditions, choices, default="e")
    df["wk_v_lower"] = np.select(lower_conditions, choices, default="e")
    cms_wk[model][prompt][dataset] = pycm.ConfusionMatrix(df["label"].tolist(), df["wk_v"].tolist(), digit=2, classes=[ 't', 'f' ])
    cms_upper[model][prompt][dataset] = pycm.ConfusionMatrix(df["label"].tolist(), df["wk_v_upper"].tolist(), digit=2, classes=[ 't', 'f' ])
    cms_lower[model][prompt][dataset] = pycm.ConfusionMatrix(df["label"].tolist(), df["wk_v_lower"].tolist(), digit=2, classes=[ 't', 'f' ])

data = [
    [ 
        model, 
        prompt, 
        dataset, 
        cms_wk[model][prompt][dataset].F1_Macro, 
        cms_wk[model][prompt][dataset].POP['t'] / len(df), 
        cms_upper[model][prompt][dataset].F1_Macro, 
        cms_upper[model][prompt][dataset].POP['t'] / len(df), 
        cms_lower[model][prompt][dataset].F1_Macro, 
        cms_lower[model][prompt][dataset].POP['t'] / len(df)
    ] 
    for model in cms 
    for prompt in cms[model] 
    for dataset in cms[model][prompt]
]

column_names = ["Judge Model", "Prompt", "Dataset", "Macro-F1", "Coverage", "Upper Macro-F1", "Upper Coverage", "Lower Macro-F1", "Lower Coverage"]
df_approx = pd.DataFrame(data, columns=column_names)
df_approx = df_approx.round(3)
df_approx = df_approx[["Dataset", "Judge Model", "Prompt", "Macro-F1", "Coverage", "Upper Macro-F1", "Lower Macro-F1"]].copy()
df_approx = df_approx.sort_values(["Dataset", "Judge Model", "Prompt"])
df_approx

,Dataset,Judge Model,Prompt,Macro-F1,Coverage,Upper Macro-F1,Lower Macro-F1
34,gpqa,claude-3-5-haiku-20241022,baseline,0.562,0.84,0.544,0.565
30,gpqa,claude-3-5-haiku-20241022,few,0.638,0.47,0.542,0.546
32,gpqa,claude-3-5-haiku-20241022,zero,0.660,0.42,0.578,0.520
23,gpqa,claude-3-5-sonnet-20241022,baseline,0.701,0.77,0.578,0.728
18,gpqa,claude-3-5-sonnet-20241022,few,0.708,0.56,0.596,0.599
20,gpqa,claude-3-5-sonnet-20241022,zero,0.717,0.51,0.576,0.613
9,gpqa,llama-4-maverick,baseline,0.786,0.85,0.728,0.760
10,gpqa,llama-4-maverick,few,0.771,0.83,0.710,0.738
6,gpqa,llama-4-maverick,zero,0.773,0.62,0.710,0.618
24,gpqa,llama-4-scout,baseline,0.763,0.72,0.653,0.720


In [25]:
approx_table_latex = df_approx.to_latex(
    index=False,          # Set to False to hide index
    float_format=lambda x: f'{x:0.3g}',
    column_format=None,  # e.g., 'lrc' for left, right, center alignment
    longtable=False,     # Set to True for tables that span multiple pages
    caption="Comparison of Macro-F1 scores of differing epistemic policies.",        # Add a caption to your table
    label="tab:expapprox",          # Add a label for cross-referencing
    position='htbp'        # e.g., 'h', 't', 'b', 'p' for table positioning
)
print(approx_table_latex, file=open("paper/approx_table.tex", "w+"))


In [26]:
df_approx.mean(numeric_only=True)

Macro-F1          0.647806
Coverage          0.581389
Upper Macro-F1    0.576306
Lower Macro-F1    0.579389
dtype: float64

In [27]:
df_approx.std(numeric_only=True)

Macro-F1          0.124073
Coverage          0.151585
Upper Macro-F1    0.067666
Lower Macro-F1    0.100467
dtype: float64

In [34]:
bi_f1 = df_approx['Macro-F1']
uni_f1 = df_approx['Upper Macro-F1']
stat, p = stats.mannwhitneyu(uni_f1, bi_f1, alternative='less', method='asymptotic')
print(f'{stat}, p = {p}')

367.5, p = 0.0008060884046759662


In [35]:
bi_f1 = df_approx['Macro-F1']
uni_f1 = df_approx['Lower Macro-F1']
stat, p = stats.mannwhitneyu(uni_f1, bi_f1, alternative='less', method='asymptotic')
print(f'{stat}, p = {p}')

410.0, p = 0.0037373228199117092


In [30]:
df_approx[df_approx["Dataset"] == "gpqa"].mean(numeric_only=True)

Macro-F1          0.645333
Coverage          0.616111
Upper Macro-F1    0.594000
Lower Macro-F1    0.581889
dtype: float64

In [31]:
df_approx[df_approx["Dataset"] == "simpleqa"].mean(numeric_only=True)

Macro-F1          0.650278
Coverage          0.546667
Upper Macro-F1    0.558611
Lower Macro-F1    0.576889
dtype: float64